# Network checks, rebuilt without Plotlay

The made-up electricity network from [getplotlay.com/showcase/network-checks](https://www.getplotlay.com/showcase/network-checks/), rebuilt from its GeoJSON with NetworkX and Shapely (`rebuild.py`), to check the answers Plotlay gave: which meters hang off transformer tr_15, which feeder supplies meter m_0283, and what loses supply when recloser sw_hv02 opens. Then Plotlay's check list against the faults planted in the data.

The network is made up. Its streets are real (Colonel Light Gardens, Adelaide, from OpenStreetMap); its feeders, transformers, switches and meters are not.

In [1]:
import json, pandas as pd
from rebuild import read, build, trace_down, trace_up, isolate
lines, points = read()
G, unjoined = build(lines, points)
print(f'{len(lines):,} lines, {len(points):,} points, joined at 0.5 m')
print('points joined to nothing:', unjoined)

2,262 lines, 1,454 points, joined at 0.5 m
points joined to nothing: ['sw_hv03', 'm_0042']


Two points join nothing: recloser sw_hv03 (planted 2.5 m off its line) and meter m_0042 (planted 1.2 m past the end of its service line). Plotlay flags both.

In [2]:
down = trace_down(G, 'tr_15')
print('tr_15 feeds', len(down), 'meters')   # Plotlay: 47

tr_15 feeds 47 meters


In [3]:
length, feeder, through = trace_up(G, 'm_0283')
print(f'm_0283 is fed from {feeder} over {length/1000:.2f} km, through {through}')   # Plotlay: FDR_CLG2, 2.14 km, tr_15 (the two map projections differ by a few metres over 2 km)

m_0283 is fed from FDR_CLG2 over 2.15 km, through ['tr_15']


In [4]:
dark = isolate(G, 'sw_hv02')
print(len(dark), 'meters lose supply with sw_hv02 open')   # Plotlay: 93

93 meters lose supply with sw_hv02 open


## Plotlay's checks against the planted faults

`data/checks/made-up-network-network-checks.csv` is the list from the Network panel's CSV button. `data/made-up-network/planted.json` lists the 15 faults put into the data. Every issue should name an id from a planted fault.

In [5]:
issues = pd.read_csv('../data/checks/made-up-network-network-checks.csv')
planted = json.load(open('../data/made-up-network/planted.json'))['planted']
def fault_of(row):
    text = f"{row['Ids']} {row['What the check found']}"
    hits = [p['n'] for p in planted if any(i in text for i in p['ids'])]
    return hits[0] if hits else None
issues['planted fault'] = issues.apply(fault_of, axis=1)
print(len(issues), 'issues;', issues['planted fault'].isna().sum(), 'not from a planted fault;', len(planted) - issues['planted fault'].nunique(), 'planted faults not found')
issues[['No.', 'Check', 'Severity', 'planted fault']]

24 issues; 0 not from a planted fault; 0 planted faults not found


,No.,Check,Severity,planted fault
0,1,Transformer attribute disagrees with the lines,error,6
1,2,Meter not fed by any transformer,error,3
2,3,Service line that joins nothing,error,3
3,4,Duplicate id,error,11
4,5,Meter joined to no line,error,2
5,6,Service line that joins nothing,error,2
6,7,Meters fed from two or more transformers,error,9
7,8,Loop in a network meant to be radial,error,9
8,9,Meter not fed by any transformer,error,4
9,10,Meter not fed by any transformer,error,4


## Gold Coast water

`data/checks/gold-coast-water-network-checks.csv` is the same list for City of Gold Coast water at Burleigh Heads (`scripts/download_gold_coast.py` fetches the data). On real data a flag is a place to look, not a fault proven on the ground.

In [6]:
w = pd.read_csv('../data/checks/gold-coast-water-network-checks.csv')
w.groupby(['Check', 'Severity']).size().sort_values(ascending=False)

Check                                        Severity
Gap closed by the tolerance                  info        993
End id disagrees with the lines              warn        156
Device joined to no main                     warn         11
Main end just short of another main          warn         10
Part of the network no supply point reaches  warn         10
Service line that joins nothing              error         3
Property on a main no supply point reaches   error         1
dtype: int64